# Entrenamiento de Red Neuronal - PyTorch

Este notebook desarrolla el proceso para entrenar una red neuronal, evaluar el sobreajuste usando TensorBoard, mitigarlo y comparar los resultados frente a los modelos clásicos (RF, KNN, Tree).

In [1]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
!pip install pandas scikit-learn tensorboard joblib statsmodels jupyter

Looking in indexes: https://download.pytorch.org/whl/cu128



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from statsmodels.stats.contingency_tables import mcnemar

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. Preparación de los Datos
Vamos a cargar los datos y preprocesarlos utilizando las mismas funciones definidas en `Funciones.py` que se utilizaron para los modelos clásicos. Así aseguramos que la comparación sea justa.

In [3]:
import sys
sys.path.append('..')
import Funciones as f

# Cargar y preprocesar
data = f.get_data('../games.csv')
data = f.filter(data, 'increment_code', 0.02)
data = f.filter(data, 'opening_eco', 0.02)

# Fijamos la semilla para reproducibilidad en el balanceo
import numpy as np
np.random.seed(42)

data = f.code(data)
data = f.balance(data)

# Recuperamos las características seleccionadas previamente
rf_data = joblib.load("../chess_random_forest_model.joblib")
list_features = list(rf_data['feature_names'])

X = data[list_features]
y = data['winner']

# Escalar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Datasets de PyTorch
train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train.values, dtype=torch.long))
test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test.values, dtype=torch.long))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Tamaño de entrenamiento: {len(train_dataset)}")
print(f"Tamaño de prueba: {len(test_dataset)}")

-----------------black -> 0------------------------
-----------------white -> 1------------------------
       white_rating  black_rating  opening_ply  victory_status_mate  \
3          0.356366      0.341805     0.166667                 True   
6          0.400435      0.325726     0.750000                False   
8          0.356366      0.309647     0.416667                False   
9          0.324810      0.214730     0.250000                 True   
13         0.324810      0.421162     0.083333                False   
...             ...           ...          ...                  ...   
20024      0.505441      0.566390     0.166667                 True   
20033      0.262786      0.237552     0.083333                False   
20049      0.295974      0.237033     0.333333                 True   
20055      0.236670      0.254668     0.166667                 True   
20057      0.245375      0.282158     0.166667                 True   

       victory_status_outoftime  victory_st

c:\Users\alvar\.gemini\antigravity\worktrees\PROYECTO_FIA\pytorch-training-analysis-report\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.5.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\alvar\.gemini\antigravity\worktrees\PROYECTO_FIA\pytorch-training-analysis-report\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.5.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Tamaño de entrenamiento: 3881
Tamaño de prueba: 971


## 2. Evaluación de Modelos Clásicos (Baseline)
Obtenemos las predicciones de los tres modelos para utilizarlas después en el Test de McNemar.

In [4]:
rf_model = rf_data['model']
knn_model = joblib.load("../k_nn_model.joblib")
tree_model = joblib.load("../decision_tree_model.joblib")

# RF y Tree se entrenaron sin escalador (pero usaremos X que es dataframe). KNN usó X escalado.
X_train_unscaled, X_test_unscaled, _, _ = train_test_split(X, y, test_size=0.2, random_state=42)

preds_rf = rf_model.predict(X_test_unscaled)
preds_knn = knn_model.predict(X_test)
preds_tree = tree_model.predict(X_test_unscaled)

print(f"Accuracy RF: {accuracy_score(y_test, preds_rf):.4f}")
print(f"Accuracy KNN: {accuracy_score(y_test, preds_knn):.4f}")
print(f"Accuracy Tree: {accuracy_score(y_test, preds_tree):.4f}")

Accuracy RF: 0.9351
Accuracy KNN: 0.5242
Accuracy Tree: 0.9186


## 3. Red Neuronal (Propenso a Sobreajuste)
Definimos una red densa y compleja sin regularización para observar el overfitting en TensorBoard.

In [5]:
class ChessNN_Overfit(nn.Module):
    def __init__(self, input_size):
        super(ChessNN_Overfit, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 2)
        )
        
    def forward(self, x):
        return self.net(x)

model_overfit = ChessNN_Overfit(X.shape[1])
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_overfit.parameters(), lr=1e-3)

writer = SummaryWriter('runs/chess_experiment_overfit')

epochs = 80
for epoch in range(epochs):
    model_overfit.train()
    total_loss, correct, total = 0, 0, 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model_overfit(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch_X.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
        
    train_loss = total_loss / total
    train_acc = correct / total
    
    model_overfit.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            outputs = model_overfit(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item() * batch_X.size(0)
            _, predicted = torch.max(outputs.data, 1)
            val_total += batch_y.size(0)
            val_correct += (predicted == batch_y).sum().item()
            
    val_loss = val_loss / val_total
    val_acc = val_correct / val_total
    
    writer.add_scalars('Loss', {'Train': train_loss, 'Validation': val_loss}, epoch)
    writer.add_scalars('Accuracy', {'Train': train_acc, 'Validation': val_acc}, epoch)

writer.close()
print("Entrenamiento finalizado. Para ver el resultado:\nAbre una terminal en esta carpeta y ejecuta: tensorboard --logdir runs")

Entrenamiento finalizado. Para ver el resultado:
Abre una terminal en esta carpeta y ejecuta: tensorboard --logdir runs


## 4. Mitigación del Sobreajuste
Introducimos Dropout y Weight Decay (regularización L2).

In [6]:
class ChessNN_Reg(nn.Module):
    def __init__(self, input_size):
        super(ChessNN_Reg, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )
        
    def forward(self, x):
        return self.net(x)

model_reg = ChessNN_Reg(X.shape[1])
optimizer_reg = optim.Adam(model_reg.parameters(), lr=1e-3, weight_decay=1e-4) # L2 Penalty
writer_reg = SummaryWriter('runs/chess_experiment_regularized')

epochs = 80
for epoch in range(epochs):
    model_reg.train()
    total_loss, correct, total = 0, 0, 0
    for batch_X, batch_y in train_loader:
        optimizer_reg.zero_grad()
        outputs = model_reg(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer_reg.step()
        
        total_loss += loss.item() * batch_X.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
        
    train_loss = total_loss / total
    train_acc = correct / total
    
    model_reg.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            outputs = model_reg(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item() * batch_X.size(0)
            _, predicted = torch.max(outputs.data, 1)
            val_total += batch_y.size(0)
            val_correct += (predicted == batch_y).sum().item()
            
    val_loss = val_loss / val_total
    val_acc = val_correct / val_total
    
    writer_reg.add_scalars('Loss', {'Train': train_loss, 'Validation': val_loss}, epoch)
    writer_reg.add_scalars('Accuracy', {'Train': train_acc, 'Validation': val_acc}, epoch)

writer_reg.close()
print("Entrenamiento con regularización completado.")

Entrenamiento con regularización completado.


## 5. Comparación Estadística (Test de McNemar)
Aplicamos el test sobre las predicciones de los modelos para determinar si las diferencias son significativas.

In [7]:
model_reg.eval()
with torch.no_grad():
    outputs = model_reg(torch.tensor(X_test, dtype=torch.float32))
    _, preds_nn = torch.max(outputs.data, 1)
preds_nn = preds_nn.numpy()

def perform_mcnemar(model_name, preds_model):
    table = [[0, 0], [0, 0]]
    for nn_pred, m_pred, true_y in zip(preds_nn, preds_model, y_test.values):
        nn_correct = (nn_pred == true_y)
        m_correct = (m_pred == true_y)
        if nn_correct and m_correct:
            table[0][0] += 1
        elif nn_correct and not m_correct:
            table[0][1] += 1
        elif not nn_correct and m_correct:
            table[1][0] += 1
        else:
            table[1][1] += 1
            
    result = mcnemar(table, exact=True)
    print(f"--- McNemar Test: Red Neuronal vs {model_name} ---")
    print(f"P-value: {result.pvalue:.5f}")
    if result.pvalue < 0.05:
        print("Diferencia estadísticamente significativa.")
    else:
        print("No hay diferencia significativa.")
    print(f"Accuracy NN: {accuracy_score(y_test, preds_nn):.4f} | Accuracy {model_name}: {accuracy_score(y_test, preds_model):.4f}\n")

perform_mcnemar("Random Forest", preds_rf)
perform_mcnemar("K-Nearest Neighbors", preds_knn)
perform_mcnemar("Decision Tree", preds_tree)

--- McNemar Test: Red Neuronal vs Random Forest ---
P-value: 0.00000
Diferencia estadísticamente significativa.
Accuracy NN: 0.6488 | Accuracy Random Forest: 0.9351

--- McNemar Test: Red Neuronal vs K-Nearest Neighbors ---
P-value: 0.00000
Diferencia estadísticamente significativa.
Accuracy NN: 0.6488 | Accuracy K-Nearest Neighbors: 0.5242

--- McNemar Test: Red Neuronal vs Decision Tree ---
P-value: 0.00000
Diferencia estadísticamente significativa.
Accuracy NN: 0.6488 | Accuracy Decision Tree: 0.9186

